# Tariikhna — Run Fine-Tuned Model (Golden Examples)

Loads your fine-tuned model from Hugging Face and generates **complete** comic
schemas for curated stories to showcase in your GP discussion.

### Pipeline for golden examples (NO Layer 1b):
```
1a  Split passage into narrative units
1c  Segment EVERY unit into scenes  (no 1b filtering - keep the whole story)
2   Generate schema (FINE-TUNED model)
3   Safety filter
```
Layer 1b (selection) is removed so no part of the story is dropped.

### Setup:
1. Runtime -> Change runtime type -> **T4 GPU**
2. Have your HF repo ready (from the upload notebook)
3. A Groq API key for Layers 1a, 1c, 3 (base model tasks)

## 1. Install

In [ ]:
!pip install unsloth groq -q
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git -q

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
# Inference needs far less than training - T4 free tier is plenty

## 2. Load the Fine-Tuned Model From Hugging Face

Loads base model + your adapters. ~5.5 GB total - fits easily on free T4.

In [ ]:
from unsloth import FastLanguageModel

# ============================================
# OPTION A: Load from Hugging Face (your uploaded repo)
# ============================================
HF_REPO = 'your-username/tariikhna-llama-3.1-8b-lora'

# If your repo is private, log in first:
# from huggingface_hub import login; login()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = HF_REPO,        # your adapters - Unsloth auto-loads the base too
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# ============================================
# OPTION B: Load from a local zip (if not using HF)
# ============================================
# from google.colab import files
# uploaded = files.upload()  # upload tariikhna_lora.zip
# import zipfile; zipfile.ZipFile(list(uploaded.keys())[0]).extractall('tariikhna_lora')
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name='tariikhna_lora', max_seq_length=2048, load_in_4bit=True)

# Switch to fast inference mode
FastLanguageModel.for_inference(model)
print('Fine-tuned model loaded and ready!')

## 3. Layer 2 Generation Function (uses YOUR fine-tuned model)

This is the layer you fine-tuned. It runs locally on the GPU - no API needed.

In [ ]:
import json, re

# Same system prompt the model was trained with
LAYER_2_SYSTEM = """You are a specialist in creating children's Islamic educational comics (ages 6-12) for the Tariikhna platform. Convert ONE visual scene into a complete schema with a detailed 250-300 word image prompt. Follow Islamic depiction rules: prophets never shown by face (use 'the central figure' or 'the man in the green cloak' in image_prompt), women modestly dressed. Output only JSON."""

def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    candidates = [i for i in [text.find('{'), text.find('[')] if i >= 0]
    if candidates:
        text = text[min(candidates):]
    try:
        return json.loads(text)
    except:
        last = text.rfind('}')
        if last > 0:
            try: return json.loads(text[:last+1])
            except: pass
        return None

def layer_2_generate(scene, source_ref=''):
    """Generate schema using the FINE-TUNED model (local GPU)."""
    messages = [
        {'role': 'system', 'content': LAYER_2_SYSTEM},
        {'role': 'user', 'content': f'Generate the schema for this scene:\n\n{json.dumps(scene, indent=2)}\n\nSource: {source_ref}'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=1024, temperature=0.6, use_cache=True
    )
    text = tokenizer.batch_decode(outputs)[0]
    # Take everything after the last assistant turn
    generated = text.split('assistant')[-1]
    return extract_json(generated)

print('Layer 2 (fine-tuned) ready!')

## 4. Layers 1a, 1c, 3 (base model via Groq)

Note: NO Layer 1b - we keep every narrative unit for complete stories.

In [ ]:
from groq import Groq
import time

GROQ_KEY = 'your-groq-key-here'
groq_client = Groq(api_key=GROQ_KEY)
GROQ_MODEL = 'llama-3.1-8b-instant'

def call_groq(system, user, max_tokens=2048, temp=0.4):
    for _ in range(5):
        try:
            r = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role':'system','content':system},{'role':'user','content':user}],
                max_tokens=max_tokens, temperature=temp)
            return r.choices[0].message.content
        except Exception as e:
            if '429' in str(e) or 'rate' in str(e).lower():
                print('  rate limited, waiting 30s'); time.sleep(30)
            else:
                print(f'  error: {str(e)[:80]}'); time.sleep(5)
    return None

LAYER_1A_SYSTEM = """You are a narrative analyst for Islamic historical texts. Split the passage into DISTINCT NARRATIVE UNITS (self-contained mini-stories). Output a JSON array; each unit has: unit_number, unit_title, unit_text, main_characters (array), event_summary. Output ONLY the JSON array."""

LAYER_1C_SYSTEM = """You are a comic storyboard artist for children's Islamic education (ages 6-12). Break ONE narrative unit into individual VISUAL SCENES (one moment = one panel). Output a JSON array; each scene has: scene_number, scene_summary, key_visual_action, characters_present (array), setting, emotional_tone, time_of_day. Output ONLY the JSON array."""

LAYER_3_SYSTEM = """You are a content safety reviewer. Rewrite the image prompt to avoid NSFW-filter trigger words WITHOUT changing the scene or removing detail. Replace: 'prophet'->'the central figure'; 'young girl/child'+body->'young woman'; 'face hidden/obscured'->'seen from behind'; 'baby/infant'+body->'a swaddled bundle'. Output JSON: {\"cleaned_prompt\": \"...\", \"changes_made\": [...]}. Output ONLY JSON."""

def layer_1a(passage):
    return extract_json(call_groq(LAYER_1A_SYSTEM, f'Split this passage:\n\n{passage}'))

def layer_1c(unit):
    return extract_json(call_groq(LAYER_1C_SYSTEM, f'Break into scenes:\n\n{json.dumps(unit, indent=2)}'))

def layer_3(image_prompt):
    r = extract_json(call_groq(LAYER_3_SYSTEM, f'Clean this prompt:\n\n{image_prompt}', max_tokens=1024, temp=0.2))
    if r and 'cleaned_prompt' in r:
        return r['cleaned_prompt'], r.get('changes_made', [])
    return image_prompt, []

print('Layers 1a, 1c, 3 ready (Layer 1b intentionally removed)!')

## 5. Story Context Generator (for the UI narration)

Generates reader-facing text: a title, an introduction, and a closing lesson.
This is the context shown alongside the images in the interface so readers
understand the story, not just see pictures.

In [ ]:
STORY_CONTEXT_SYSTEM = """You are writing reader-facing text for a children's Islamic comic (ages 6-12).
Given a story passage, produce warm, simple, engaging framing text for young readers.

Output JSON:
{
  "story_title": "a warm, child-friendly title (not the academic heading)",
  "introduction": "2-3 simple sentences setting up the story, drawing the child in",
  "conclusion": "2-3 sentences closing the story warmly",
  "moral_lesson": "one clear lesson a child can take away, in simple words",
  "reading_age": "6-9 or 8-12",
  "key_figures": ["list of main people in the story"]
}
Keep language simple, warm, and age-appropriate. Output ONLY the JSON."""

def generate_story_context(passage, source_ref=''):
    """Generate the reader-facing framing text for the whole story."""
    raw = call_groq(STORY_CONTEXT_SYSTEM, f'Write reader framing for this story:\n\n{passage}', max_tokens=1024, temp=0.6)
    return extract_json(raw)

print('Story context generator ready!')

## 6. Golden-Example Pipeline (1a -> 1c -> 2 -> 3, every unit kept)

Now also generates story context so each panel image has reader text alongside it.

In [ ]:
def generate_golden_story(passage, source_ref='', verbose=True):
    """Generate a COMPLETE story with reader context - no unit dropped (no Layer 1b)."""
    result = {'passage': passage, 'source': source_ref, 'story_context': None, 'units': None, 'panels': []}
    
    # Story-level context first (for the UI narration)
    if verbose: print('Generating story context (title, intro, lesson)...')
    result['story_context'] = generate_story_context(passage, source_ref)
    if result['story_context'] and verbose:
        print(f'  Title: {result["story_context"].get("story_title", "?")}')
    time.sleep(2)
    
    if verbose: print('Layer 1a: splitting into units...')
    units = layer_1a(passage)
    if not units:
        print('  FAILED at 1a'); return result
    result['units'] = units
    if verbose: print(f'  {len(units)} units found (ALL kept)')
    time.sleep(2)
    
    panel_number = 0
    # NO Layer 1b - iterate over EVERY unit
    for u_idx, unit in enumerate(units):
        if verbose: print(f'\nUnit {u_idx+1}: {unit.get("unit_title", "?")}')
        scenes = layer_1c(unit)
        if not scenes:
            if verbose: print('  segmentation failed, skipping unit')
            continue
        if verbose: print(f'  {len(scenes)} scenes')
        time.sleep(2)
        
        for scene in scenes:
            schema = layer_2_generate(scene, source_ref)
            if not schema:
                if verbose: print('    scene failed at layer 2')
                continue
            if 'image_prompt' in schema:
                cleaned, changes = layer_3(schema['image_prompt'])
                schema['image_prompt'] = cleaned
                schema['_safety_changes'] = changes
            
            panel_number += 1
            # Each panel carries: the image prompt AND the reader-facing narration
            result['panels'].append({
                'panel_number': panel_number,
                'unit_title': unit.get('unit_title', ''),
                'scene_input': scene,
                'narrative_text': schema.get('narrative_text', ''),   # <- reader text for UI
                'moral_lesson': schema.get('moral_lesson', ''),
                'image_prompt': schema.get('image_prompt', ''),
                'characters': schema.get('characters', []),
                'compliance': schema.get('compliance', {}),
                'full_schema': schema
            })
            if verbose:
                print(f'    Panel {panel_number}: {schema.get("scene_title","?")}')
    
    if verbose: print(f'\nComplete story: {len(result["panels"])} panels with narration')
    return result

print('Golden-example pipeline ready (with reader context)!')

## 7. Select & Generate Your Golden Stories

Four curated complete-episode stories selected for the GP discussion.

In [ ]:
# Load the 4 selected golden passages (upload golden_passages_selected.json)
import json
with open('golden_passages_selected.json', encoding='utf-8') as f:
    GOLDEN_PASSAGES = json.load(f)

print(f'{len(GOLDEN_PASSAGES)} golden stories loaded:')
for p in GOLDEN_PASSAGES:
    print(f'  {p["display_title"]} ({len(p["passage"].split())}w)')

In [ ]:
import os
os.makedirs('golden_output', exist_ok=True)

golden_results = []
for i, p in enumerate(GOLDEN_PASSAGES):
    print(f'\n{"="*60}')
    print(f'GOLDEN STORY {i+1}/{len(GOLDEN_PASSAGES)}: {p["display_title"]}')
    print(f'{"="*60}')
    res = generate_golden_story(p['passage'], p.get('source', ''), verbose=True)
    res['passage_id'] = p['passage_id']
    res['display_title'] = p['display_title']
    golden_results.append(res)
    with open(f'golden_output/{p["passage_id"]}.json', 'w', encoding='utf-8') as f:
        json.dump(res, f, ensure_ascii=False, indent=2)

print(f'\n\nDONE! {len(golden_results)} stories, {sum(len(r["panels"]) for r in golden_results)} total panels.')

## 8. Inspect: Each Panel With Its Reader Narration

In [ ]:
for res in golden_results:
    ctx = res.get('story_context') or {}
    print(f'\n{"="*65}')
    print(f'STORY: {ctx.get("story_title", res["display_title"])}')
    print(f'{"="*65}')
    print(f'Introduction: {ctx.get("introduction", "")}')
    print(f'Lesson: {ctx.get("moral_lesson", "")}')
    print(f'Panels: {len(res["panels"])}')
    print('-'*65)
    for panel in res['panels']:
        print(f'\n  PANEL {panel["panel_number"]}')
        print(f'  Reader text: {panel["narrative_text"]}')
        print(f'  Image prompt ({len(panel["image_prompt"].split())}w): {panel["image_prompt"][:160]}...')
    print(f'\nConclusion: {ctx.get("conclusion", "")}')

In [ ]:
# Download everything
import shutil
shutil.make_archive('golden_output', 'zip', 'golden_output')
from google.colab import files
files.download('golden_output.zip')
print('Downloaded golden_output.zip')
print('Each story JSON now has: story_context (title/intro/lesson/conclusion) + panels (each with narrative_text + image_prompt)')